# Chainsaw Man Gesture Recognition - Training Notebook

This notebook trains a custom gesture recognizer for Contract and Hybrid gestures using MediaPipe Model Maker.

**Important:** Upload your training data to Colab first:
- data/gestures/contract/ (100+ images)
- data/gestures/hybrid/ (100+ images)
- data/gestures/none/ (100+ images of non-gestures)


## 1. Install Dependencies

In [ ]:
!pip install --upgrade pip
!pip install mediapipe-model-maker

## 2. Import Libraries

In [ ]:
from google.colab import files
import os
import tensorflow as tf
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer
import matplotlib.pyplot as plt

## 3. Upload Your Training Data

Run this cell and upload your data folder (data/gestures/)

In [ ]:
# Upload your dataset
print("Select your data/gestures/ folder to upload...")
uploaded = files.upload()

# If you uploaded a zip file, extract it
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        !unzip -q {filename}
        print(f"Extracted {filename}")

## 4. Verify Dataset Structure

In [ ]:
dataset_path = "gestures"  # Update if different

print(f"Dataset path: {dataset_path}")
print(f"\nDataset contents:")

labels = []
for item in os.listdir(dataset_path):
    path = os.path.join(dataset_path, item)
    if os.path.isdir(path):
        num_images = len(os.listdir(path))
        labels.append(item)
        print(f"  {item}: {num_images} images")

print(f"\nLabels found: {labels}")
print(f"\n✓ IMPORTANT: Must include 'none' label: {'none' in labels}")

## 5. Visualize Sample Images

In [ ]:
NUM_EXAMPLES = 5

for label in labels:
    label_dir = os.path.join(dataset_path, label)
    example_filenames = os.listdir(label_dir)[:NUM_EXAMPLES]
    fig, axs = plt.subplots(1, NUM_EXAMPLES, figsize=(15, 3))
    for i in range(NUM_EXAMPLES):
        img_path = os.path.join(label_dir, example_filenames[i])
        axs[i].imshow(plt.imread(img_path))
        axs[i].get_xaxis().set_visible(False)
        axs[i].get_yaxis().set_visible(False)
    fig.suptitle(f'{NUM_EXAMPLES} examples for {label} gesture')
    plt.tight_layout()
    plt.show()

## 6. Load and Split Dataset

Loads dataset from folder, extracts hand landmarks using MediaPipe Hands, and splits into train/validation/test sets.

In [ ]:
print("Loading dataset and extracting hand landmarks...")
print("(Images without detected hands will be skipped)\n")

data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)

print("Splitting dataset: 80% train, 10% validation, 10% test...")
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

print(f"\n✓ Training samples: {len(train_data)}")
print(f"✓ Validation samples: {len(validation_data)}")
print(f"✓ Test samples: {len(test_data)}")

## 7. Train the Model

This will take 5-10 minutes depending on dataset size.

In [ ]:
print("Training custom gesture recognizer...\n")

hparams = gesture_recognizer.HParams(export_dir="exported_model")
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)

model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

print("\n✓ Training complete!")

## 8. Evaluate Model Performance

In [ ]:
print("Evaluating model on test set...\n")

loss, acc = model.evaluate(test_data, batch_size=1)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")

## 9. Export Model

In [ ]:
print("Exporting model...\n")

model.export_model()

print("\nExported files:")
!ls -lh exported_model/

## 10. Download the Model

In [ ]:
print("Downloading gesture_recognizer.task...\n")
files.download('exported_model/gesture_recognizer.task')
print("\n✓ Download complete!")
print("\nMove this file to your local project:")
print("  mv gesture_recognizer.task ~/chainsaw_man_gesture_recognition/models/")

## 11. (Optional) Fine-tune with Hyperparameters

Train another model with different settings to compare.

In [ ]:
# Uncomment to try different hyperparameters
# hparams = gesture_recognizer.HParams(
#     learning_rate=0.003,
#     epochs=15,
#     batch_size=4,
#     export_dir="exported_model_v2"
# )
# model_options = gesture_recognizer.ModelOptions(dropout_rate=0.2)
# options = gesture_recognizer.GestureRecognizerOptions(
#     model_options=model_options,
#     hparams=hparams
# )
# 
# model_2 = gesture_recognizer.GestureRecognizer.create(
#     train_data=train_data,
#     validation_data=validation_data,
#     options=options
# )
# 
# loss_2, acc_2 = model_2.evaluate(test_data)
# print(f"Model 2 - Loss: {loss_2:.4f}, Accuracy: {acc_2:.4f}")